# Interpretability and Mechanistic Interpretability

দুটি স্বয়ংসম্পূর্ণ, সৎ demonstration:

**PART 1 — Probing classifiers.**
একটি ছোট MLP-কে একটি **PRIMARY** task-এ train করা হয় (একটি 8-bit binary সংখ্যাকে 4টি magnitude bucket-এর একটিতে শ্রেণিবদ্ধ করা — এমন একটি task যা ঠিক দুটি most-significant bit দিয়েই সম্পূর্ণ নির্ধারিত)। Least-significant bit — parity, অর্থাৎ জোড়/বিজোড় — bucket label থেকে statistically independent এবং training signal-এর অংশ কখনোই নয়। তারপর trained model-টিকে freeze করে, এর internal hidden activation-গুলো বের করে, সেই frozen activation থেকে parity পুনরুদ্ধার করতে একটি linear probe train করা হয়। Trained model-এর probe accuracy-কে দুটি baseline-এর সাথে তুলনা করা হয়: raw input bit-এর উপর সরাসরি train করা একটি probe, এবং একই architecture-এর একটি **UNTRAINED** (random-weight) copy-র activation-এর উপর train করা একটি probe। উদ্দেশ্য হলো README-র "probing representation প্রমাণ করে, causal use নয়" সতর্কবার্তাটিকে বাস্তব করে তোলা: আমরা সরাসরি যাচাই করতে পারি trained model-এর নিজস্ব **OUTPUT** আসলে parity-র উপর নির্ভর করে কি না (নির্মাণগতভাবেই করে না), এমনকি যেখানে একটি probe এর activation-এ parity-কে linearly decodable খুঁজে পায়।

**PART 2 — Sparse Autoencoders (SAEs) এবং superposition.**
Synthetic "activation-সদৃশ" vector তৈরি করা হয় বেশি সংখ্যক প্রকৃত অন্তর্নিহিত feature (30টি)-কে feature-এর চেয়ে কম raw dimension (10টি)-এ ঠেসে ঢুকিয়ে — একটি standard toy superposition setup — নির্দিষ্ট feature direction-গুলোর sparse random combination-এর মাধ্যমে। একটি ছোট overcomplete SAE (hidden dimension >> input dimension) কয়েকটি ভিন্ন L1 sparsity penalty strength-এ এই vector-গুলোর উপর train করা হয়, এবং এর ফলে তৈরি হওয়া reconstruction-quality বনাম sparsity/dead-unit trade-off প্রকৃত মাপা সংখ্যা দিয়ে রিপোর্ট করা হয়।

Runtime: CPU-তে এক মিনিটেরও অনেক কম।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। এই script-এ কোনো আলাদা demo function বা `main()` নেই — প্রতিটি cell-এর code সরাসরি চলে এবং পরের cell আগের cell-এর variable ব্যবহার করে।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1.1 PART 1 — Probing classifiers: toy data

0..255 পর্যন্ত প্রতিটি integer-কে একটি 8-bit binary vector হিসেবে লেখা হয় (MSB আগে)। প্রথম দুটি bit মিলে "bucket" label (`value // 64`, অর্থাৎ 4টি bucket) নির্ধারণ করে, আর শেষ bit-টি হুবহু parity bit। যেহেতু uniform 0..255 integer-এর 8টি bit পরস্পর independent, তাই bucket label ও parity label নির্মাণগতভাবেই statistically independent। Data একবার shuffle করে train/test-এ ভাগ করা হয়, যাতে দুটি task একই split ব্যবহার করে।

In [ ]:
# =============================================================================
# PART 1 -- Probing classifiers
# =============================================================================

print("=" * 78)
print("PART 1 -- PROBING CLASSIFIERS")
print("=" * 78)

# ---------------------------------------------------------------------------
# 1.1 Toy data: 0..255 পর্যন্ত প্রতিটি integer একটি 8-bit binary vector হিসেবে, MSB আগে।
#     bit index 0 (সবচেয়ে বামের, most significant) এবং bit index 1 মিলে হুবহু
#     "bucket" label নির্ধারণ করে (value // 64, তাই 4টি bucket)।
#     bit index 7 (সবচেয়ে ডানের, least significant) হুবহু parity bit
#     (জোড়/বিজোড়)। যেহেতু একটি uniform 0..255 integer-এর 8টি bit-ই পরস্পর
#     independent, তাই bucket label এবং parity label নির্মাণগতভাবেই পরস্পরের থেকে
#     STATISTICALLY INDEPENDENT -- primary task-এর প্রয়োজনীয় কোনো তথ্য parity বহন করে না।
# ---------------------------------------------------------------------------

def to_bits(n, num_bits=8):
    return [float((n >> (num_bits - 1 - i)) & 1) for i in range(num_bits)]


all_values = list(range(256))
inputs = torch.tensor([to_bits(v) for v in all_values], dtype=torch.float32)   # (256, 8)
buckets = torch.tensor([v // 64 for v in all_values], dtype=torch.long)        # (256,), মান {0,1,2,3}-এ
parity = torch.tensor([v % 2 for v in all_values], dtype=torch.long)           # (256,), মান {0,1}-এ, 0=জোড় 1=বিজোড়

# একবার shuffle করে তারপর train/test-এ ভাগ, যাতে দুটি task একই split ভাগ করে নেয়।
perm = torch.randperm(256)
train_idx, test_idx = perm[:200], perm[200:]

x_train, x_test = inputs[train_idx], inputs[test_idx]
bucket_train, bucket_test = buckets[train_idx], buckets[test_idx]
parity_train, parity_test = parity[train_idx], parity[test_idx]

print(f"Dataset: all 256 8-bit integers -> {len(train_idx)} train / {len(test_idx)} test")
print("Primary task label (bucket) depends only on the top 2 bits.")
print("Probed concept (parity) depends only on the bottom 1 bit.")
print("These two labels are statistically independent by construction.\n")

## 1.2 শুধু PRIMARY task-এ train করা model এবং causal check

`BucketClassifier` একটি ছোট MLP যা `hidden2` (16-dim) activation-ও ফেরত দেয় — এটিকেই পরে probe করা হবে। Model-টিকে শুধু bucket classification-এ train করা হয়, তারপর একটি causal check চালানো হয়: প্রতিটি test input-এর parity bit উল্টে দিলে model-এর prediction কতবার বদলায়। এটিই সেই ground truth যার সাথে পরে probe-এর ফলাফল তুলনা করা হবে।

In [ ]:
# ---------------------------------------------------------------------------
# 1.2 শুধু PRIMARY task-এ (bucket classification) train করা model।
#     আমরা `hidden2` (16-dim) থেকে বের হওয়া activation-গুলো probe করব।
# ---------------------------------------------------------------------------

class BucketClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(8, 32)
        self.hidden2 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 4)

    def forward(self, x):
        h1 = F.relu(self.hidden1(x))
        h2 = F.relu(self.hidden2(h1))     # <-- এটিই সেই activation যা আমরা probe করি
        logits = self.output(h2)
        return logits, h2


trained_model = BucketClassifier()
optimizer = torch.optim.Adam(trained_model.parameters(), lr=0.02)

for epoch in range(300):
    logits, _ = trained_model(x_train)
    loss = F.cross_entropy(logits, bucket_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_logits, _ = trained_model(x_test)
    bucket_acc = (test_logits.argmax(dim=-1) == bucket_test).float().mean().item()
print(f"Primary task (bucket classification) test accuracy: {bucket_acc * 100:.1f}%")

# Sanity/causal check: trained model-এর নিজস্ব prediction কি আসলে parity-র উপর
# নির্ভর করে? প্রতিটি test input-এর least-significant bit উল্টে দিয়ে দেখা হয়
# predicted bucket বদলায় কি না। Model যদি task-এর চাহিদা অনুযায়ী কাজ করে, তাহলে
# parity উল্টালে prediction ~0% ক্ষেত্রে বদলানো উচিত -- এটিই সেই ground truth
# যার সাথে আমরা probe-এর ফলাফল তুলনা করব।
with torch.no_grad():
    x_test_flipped = x_test.clone()
    x_test_flipped[:, -1] = 1.0 - x_test_flipped[:, -1]   # শুধু parity bit উল্টানো
    logits_orig, _ = trained_model(x_test)
    logits_flipped, _ = trained_model(x_test_flipped)
    pred_changed = (logits_orig.argmax(-1) != logits_flipped.argmax(-1)).float().mean().item()
print(f"Fraction of predictions that change when we flip parity (causal check): {pred_changed * 100:.1f}%")
print("-> The model's own OUTPUT does not use parity at all (as expected -- it")
print("   was never part of the training signal). Any probe that nonetheless")
print("   decodes parity from an internal LAYER is finding a REPRESENTATION,")
print("   not evidence the model's computation actually USES it downstream.")

## 1.3 Frozen activation বের করা এবং linear probe train করা

তিনটি উৎস থেকে frozen activation (classifier-এ কোনো gradient যায় না) বের করে প্রতিটির উপর parity অনুমান করতে একটি linear probe train করা হয়: (a) trained model-এর `hidden2` activation, (b) raw input bit (baseline), এবং (c) একই architecture-এর একটি untrained copy-র `hidden2` activation (baseline)। Probe ইচ্ছাকৃতভাবে মাত্র একটি linear layer, যাতে সাফল্য মানে concept-টি **linearly** decodable। শেষে প্রকৃত সংখ্যাগুলোর ব্যাখ্যা print করা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1.3 তিনটি উৎস থেকে FROZEN activation বের করা (classifier-এ কোনো gradient নয়),
#     তারপর প্রতিটির উপর parity অনুমান করতে একটি linear probe train করা:
#       (a) TRAINED model-এর hidden2 activation
#       (b) raw input bit নিজেই (baseline: concept-টি কি আদৌ input-এ
#           linearly উপস্থিত?)
#       (c) হুবহু একই architecture-এর একটি UNTRAINED (সদ্য initialize করা, কখনো
#           train করা হয়নি) copy-র hidden2 activation (baseline: শুধু random
#           projection-ই কি linear decodability ধরে রাখে?)
# ---------------------------------------------------------------------------

def get_frozen_activations(model, x):
    model.eval()
    with torch.no_grad():
        _, h2 = model(x)
    return h2


trained_model.eval()
h_train_trained = get_frozen_activations(trained_model, x_train)
h_test_trained = get_frozen_activations(trained_model, x_test)

untrained_model = BucketClassifier()   # সদ্য initialize করা, কখনো train করা হয়নি
h_train_untrained = get_frozen_activations(untrained_model, x_train)
h_test_untrained = get_frozen_activations(untrained_model, x_test)


def train_linear_probe(feat_train, label_train, feat_test, label_test, num_classes=2, epochs=400, lr=0.05):
    """একটি probe ইচ্ছাকৃতভাবে মাত্র একটি linear layer -- logistic regression --
    যাতে সাফল্যের অর্থ হয় concept-টি LINEARLY decodable, কোনো ইচ্ছামতো শক্তিশালী
    nonlinear function দিয়ে decodable নয় (যা প্রায় যেকোনো representation থেকে
    প্রায় যেকোনো কিছু পুনরুদ্ধার করতে পারত এবং আমাদের খুব কমই কিছু বলত)।"""
    probe = nn.Linear(feat_train.shape[1], num_classes)
    opt = torch.optim.Adam(probe.parameters(), lr=lr)
    feat_train = feat_train.detach()
    feat_test = feat_test.detach()
    for _ in range(epochs):
        logits = probe(feat_train)
        loss = F.cross_entropy(logits, label_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        test_acc = (probe(feat_test).argmax(-1) == label_test).float().mean().item()
    return test_acc


acc_raw_input = train_linear_probe(x_train, parity_train, x_test, parity_test)
acc_trained_activations = train_linear_probe(h_train_trained, parity_train, h_test_trained, parity_test)
acc_untrained_activations = train_linear_probe(h_train_untrained, parity_train, h_test_untrained, parity_test)

print("\nLinear probe test accuracy for the PARITY concept, by feature source:")
print(f"  (a) probe on TRAINED model's hidden activations   : {acc_trained_activations * 100:5.1f}%")
print(f"  (b) probe on RAW INPUT bits (baseline)             : {acc_raw_input * 100:5.1f}%")
print(f"  (c) probe on UNTRAINED (random) model's activations: {acc_untrained_activations * 100:5.1f}%")


def describe(acc, name):
    if acc > 0.95:
        return f"near-perfect -- {name} still linearly encodes parity almost exactly"
    elif acc > 0.65:
        return f"clearly above chance (50%) -- {name} partially preserves parity"
    else:
        return f"close to chance (50%) -- {name} has largely destroyed parity"


print("\nInterpretation of these actual numbers:")
print(f"  - Raw input probe is {describe(acc_raw_input, 'the raw input')}: parity IS")
print("    exactly one input coordinate, so a linear probe can read it off directly.")
print(f"  - Trained-model probe is {describe(acc_trained_activations, 'the trained activations')}.")
print(f"  - Untrained-model probe is {describe(acc_untrained_activations, 'the untrained activations')}.")
if acc_untrained_activations > 0.65:
    print("  - Honest note: even an UNTRAINED network's random projection keeps parity")
    print("    substantially linearly readable here -- a real, known phenomenon (random")
    print("    features are often somewhat linearly informative). So high probe accuracy")
    print("    on the trained model, by itself, is only weak evidence that TRAINING")
    print("    specifically is what makes the concept decodable -- an untrained network")
    print("    can look similarly decodable. Combined with the causal check above (the")
    print("    model's own predictions do not change when parity is flipped), this is a")
    print("    concrete illustration of the lesson's central caveat: a probe finding a")
    print("    linearly-readable concept is evidence about REPRESENTATION, not proof the")
    print("    model's downstream computation CAUSALLY USES that concept.")
else:
    print("  - The untrained network's activations are much less decodable than the")
    print("    trained model's, which is consistent with a story where training itself")
    print("    reshapes the representation. Even so, remember the causal check above:")
    print("    the trained model's OWN predictions never actually use parity -- so this")
    print("    probe result is still only evidence of REPRESENTATION, not causal use.")

## 2.1 PART 2 — Sparse Autoencoders এবং superposition: synthetic data

Superposition-সহ synthetic "activation-সদৃশ" data তৈরি করা হয়: মাত্র 10-dimensional একটি ambient space-এ 30টি নির্দিষ্ট, random, unit-norm feature direction — raw dimension-এর চেয়ে বেশি প্রকৃত feature, ঠিক Elhage et al. (2022)-এর toy-superposition setup-এর মতো। প্রতিটি sample 30টি feature-এর শুধু একটি sparse random subset (গড়ে ~3টি) সক্রিয় করে এবং তাদের direction-গুলো random nonnegative coefficient দিয়ে scale করে একটি 10-dim vector-এ যোগ করে।

In [ ]:
# =============================================================================
# PART 2 -- Sparse Autoencoders এবং superposition
# =============================================================================

print("\n" + "=" * 78)
print("PART 2 -- SPARSE AUTOENCODERS (SAE) AND SUPERPOSITION")
print("=" * 78)

# ---------------------------------------------------------------------------
# 2.1 Superposition আগে থেকেই বসানো synthetic "activation-সদৃশ" data তৈরি:
#     N_TRUE_FEATURES (30)টি নির্দিষ্ট, random, unit-norm feature direction, যারা
#     মাত্র AMBIENT_DIM (10) dimension-এর একটি AMBIENT space-এ থাকে -- raw
#     dimension-এর চেয়ে বেশি প্রকৃত feature, হুবহু Elhage et al. (2022)-এর
#     toy-superposition setup। প্রতিটি sample 30টি feature-এর শুধু একটি SPARSE
#     random subset (গড়ে ~3টি) সক্রিয় করে, তারপর তাদের direction-গুলোকে random
#     nonnegative coefficient দিয়ে scale করে একটি 10-dim vector-এ যোগ করে।
#     ধারণা করা হয় একটি প্রকৃত model-এর activation দেখতে এরকমই: raw neuron-এর
#     চেয়ে অনেক বেশি "প্রকৃত concept", sparsity-র মাধ্যমে superimposed।
# ---------------------------------------------------------------------------

N_TRUE_FEATURES = 30
AMBIENT_DIM = 10
SPARSITY_PROB = 0.1     # প্রতি sample-এ প্রতিটি প্রকৃত feature "সক্রিয়" হওয়ার সম্ভাবনা (গড়ে 30-এর মধ্যে ~3টি)
N_SAMPLES = 4000

feature_directions = F.normalize(torch.randn(AMBIENT_DIM, N_TRUE_FEATURES), dim=0)   # unit vector, column হিসেবে

active_mask = (torch.rand(N_SAMPLES, N_TRUE_FEATURES) < SPARSITY_PROB).float()
magnitudes = torch.rand(N_SAMPLES, N_TRUE_FEATURES) * 0.5 + 0.5   # সক্রিয় হলে magnitude [0.5, 1.0]-এর মধ্যে
coefficients = active_mask * magnitudes                            # (N_SAMPLES, N_TRUE_FEATURES), বেশিরভাগই শূন্য

activation_data = coefficients @ feature_directions.T               # (N_SAMPLES, AMBIENT_DIM)

avg_active_true_features = active_mask.sum(dim=1).mean().item()
print(f"Synthetic data: {N_TRUE_FEATURES} true features superimposed into {AMBIENT_DIM} raw dimensions")
print(f"Average number of true features active per sample: {avg_active_true_features:.2f} (of {N_TRUE_FEATURES})")

data_train, data_test = activation_data[:3200], activation_data[3200:]

## 2.2 SAE নিজেই: বিভিন্ন L1 strength-এ training এবং trade-off

SAE-টি হলো linear encoder -> ReLU -> linear decoder, এবং **overcomplete**: hidden dim (60) input dim (10)-এর চেয়ে বড়, যাতে superimposed feature-গুলোকে আলাদা direction-এ খুলে ফেলার জায়গা থাকে। `train_sae` reconstruction loss ও L1 sparsity penalty দিয়ে SAE train করে, তারপর test set-এ reconstruction MSE, গড় sparsity, এবং alive/dead unit-এর সংখ্যা মাপে। দুর্বল থেকে শক্তিশালী — চারটি L1 strength-এ এটি চালিয়ে প্রকৃত সংখ্যা ও তাদের ব্যাখ্যা print করা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 2.2 SAE নিজেই: linear encoder -> ReLU -> linear decoder।
#     OVERCOMPLETE: hidden dim (60) input dim (10)-এর চেয়ে বড়, তাই superimposed
#     feature-গুলোকে আলাদা আলাদা direction-এ খুলে ফেলার জায়গা model-এর থাকে।
# ---------------------------------------------------------------------------

SAE_HIDDEN_DIM = 60   # overcomplete: ambient input dimension-এর 6 গুণ
ACTIVE_THRESHOLD = 0.05   # কোনো sample-এ একটি hidden unit-এর মান এর বেশি হলে সেটি "সক্রিয়" গণ্য হয়


class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        h = F.relu(self.encoder(x))     # sparse code (training বেশিরভাগ unit-কে 0-তে ঠেলে দেওয়ার পর)
        x_hat = self.decoder(h)
        return x_hat, h


def train_sae(l1_strength, epochs=300, lr=0.02):
    sae = SparseAutoencoder(AMBIENT_DIM, SAE_HIDDEN_DIM)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    for _ in range(epochs):
        x_hat, h = sae(data_train)
        recon_loss = F.mse_loss(x_hat, data_train)
        sparsity_loss = h.abs().mean()          # batch ও hidden unit জুড়ে গড় L1
        loss = recon_loss + l1_strength * sparsity_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

    sae.eval()
    with torch.no_grad():
        x_hat_test, h_test = sae(data_test)
        final_recon_loss = F.mse_loss(x_hat_test, data_test).item()
        active_per_sample = (h_test > ACTIVE_THRESHOLD).float()          # (N_test, SAE_HIDDEN_DIM)
        avg_sparsity = active_per_sample.mean().item()                    # প্রতি input-এ সক্রিয় unit-এর গড় ভগ্নাংশ
        # একটি unit "alive" যদি এটি অন্তত 1% test sample-এ fire করে, নইলে "dead"
        fraction_of_samples_active = active_per_sample.mean(dim=0)        # প্রতি unit-এর firing rate
        num_alive_units = (fraction_of_samples_active > 0.01).sum().item()
        num_dead_units = SAE_HIDDEN_DIM - num_alive_units
    return final_recon_loss, avg_sparsity, num_alive_units, num_dead_units


L1_STRENGTHS = [0.0005, 0.005, 0.02, 0.08]   # দুর্বল -> শক্তিশালী

print(f"\nTraining a {AMBIENT_DIM} -> {SAE_HIDDEN_DIM} (overcomplete) SAE at {len(L1_STRENGTHS)} L1 strengths...\n")
print(f"{'L1 strength':>12} | {'recon. MSE':>10} | {'avg sparsity':>12} | {'alive units':>12} | {'dead units':>10}")
print("-" * 68)

results = []
for l1 in L1_STRENGTHS:
    recon_loss, avg_sparsity, num_alive, num_dead = train_sae(l1)
    results.append((l1, recon_loss, avg_sparsity, num_alive, num_dead))
    print(f"{l1:>12} | {recon_loss:>10.4f} | {avg_sparsity:>11.1%} | {num_alive:>12d} | {num_dead:>10d}")

print("\nInterpretation of these actual numbers:")
weakest = results[0]
strongest = results[-1]
print(f"  - Weakest L1 ({weakest[0]}): reconstruction MSE = {weakest[1]:.4f}, "
      f"avg sparsity = {weakest[2]:.1%}, {weakest[4]} dead units out of {SAE_HIDDEN_DIM}.")
print(f"  - Strongest L1 ({strongest[0]}): reconstruction MSE = {strongest[1]:.4f}, "
      f"avg sparsity = {strongest[2]:.1%}, {strongest[4]} dead units out of {SAE_HIDDEN_DIM}.")

if strongest[2] < weakest[2] and strongest[4] >= weakest[4]:
    print("  - As predicted: increasing the L1 strength drives the code SPARSER (lower")
    print("    average fraction of active units per input) and pushes MORE units to")
    print("    become permanently dead, while weaker L1 keeps more units alive but with")
    print("    denser, less individually-selective activations.")
if strongest[1] > weakest[1]:
    print("  - And reconstruction quality trades off against that sparsity: the strongest")
    print("    L1 setting has the WORST (highest) reconstruction error of the settings")
    print("    tested here -- squeezing the code sparser makes it a lossier bottleneck.")
else:
    print("  - Reconstruction error did not strictly worsen with stronger L1 in this run --")
    print("    with this much data and this small a feature set the network still found a")
    print("    reasonably faithful sparse code even under the strongest penalty tested.")

print("\n-> This is the real reconstruction/sparsity/dead-unit trade-off SAE research")
print("   has to navigate at every scale: enough sparsity pressure to pull apart")
print("   superimposed, polysemantic directions into individually monosemantic")
print("   features, without so much pressure that reconstruction (and therefore the")
print("   features actually being preserved at all) falls apart or units simply die.")